# Cenário 12: Renda Vitalícia de Profissões Essenciais

## Contexto
Profissões como enfermagem, magistério e medicina são pilares da sociedade. Mas quanto um desses profissionais acumularia em uma vida inteira de trabalho, comparado à fortuna de Musk?

**Fontes:**
- [CFE – Piso de Enfermagem (Lei 14.434/2022)](https://www.gov.br)
- [MEC – Piso Nacional do Magistério](https://www.gov.br/mec)
- [CFM – Pesquisa de Remuneração Médica](https://www.cfm.org.br)
- [Forbes Billionaires – Elon Musk](https://www.forbes.com/real-time-billionaires/)

In [1]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

MUSK_USD   = 1_000_000_000_000   # US$ 1 trilhão (Forbes, jun/2026)
MUSK_BRL   = MUSK_USD * 5.71    # câmbio aproximado jun/2026
USD_BRL    = 5.71
DATA       = 'data/'
prof = pd.read_csv(DATA + "profissoes_brasil.csv")
anos_trabalho = 35  # vida laborativa típica

prof["renda_anual_brl"]    = prof["salario_medio_brl"] * 12
prof["renda_vitalicia_brl"]= prof["renda_anual_brl"] * anos_trabalho
prof["renda_vitalicia_usd"]= prof["renda_vitalicia_brl"] / USD_BRL
prof["n_vidas_para_musk"]  = MUSK_USD / prof["renda_vitalicia_usd"]
prof["pct_fortuna_musk"]   = prof["renda_vitalicia_usd"] / MUSK_USD * 100

print("Renda vitalícia vs. Fortuna de Musk (35 anos de trabalho):")
print(prof[["profissao", "salario_medio_brl", "renda_vitalicia_brl", "n_vidas_para_musk"]].to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Gráfico 1: Número de vidas laborais necessárias
prof_s = prof.sort_values('n_vidas_para_musk', ascending=True)
cores_h = plt.cm.Reds(np.linspace(0.3, 0.9, len(prof_s)))
hbars = axes[0].barh(prof_s["profissao"], prof_s["n_vidas_para_musk"],
                     color=cores_h, edgecolor="white", height=0.6)
axes[0].set_xlabel("Nº de vidas laborais necessárias (35 anos cada)", fontsize=11)
axes[0].set_title("Quantas Vidas de Trabalho Equivalem\nà Fortuna de Musk?",
                  fontsize=12, fontweight="bold")
axes[0].set_xscale("log")
for bar, val in zip(hbars, prof_s["n_vidas_para_musk"]):
    if val >= 1e6:
        label = f"{val/1e6:.1f}M vidas"
    else:
        label = f"{val/1e3:.0f}mil vidas"
    axes[0].text(val * 1.05, bar.get_y() + bar.get_height()/2,
                 label, va="center", fontsize=9)

# Gráfico 2: Renda vitalícia em BRL
prof_s2 = prof.sort_values("renda_vitalicia_brl")
cores_h2 = plt.cm.Blues(np.linspace(0.3, 0.9, len(prof_s2)))
hbars2 = axes[1].barh(prof_s2["profissao"], prof_s2["renda_vitalicia_brl"]/1e6,
                      color=cores_h2, edgecolor="white", height=0.6)
musk_brl_bi = MUSK_BRL / 1e6
axes[1].axvline(x=musk_brl_bi, color="#e74c3c", linestyle="--", linewidth=2,
                label=f"Fortuna de Musk (R$ {MUSK_BRL/1e12:.2f} tri)")
axes[1].set_xlabel("Renda vitalícia (milhões BRL)", fontsize=11)
axes[1].set_title("Renda Vitalícia (35 anos) por Profissão\nvs. Fortuna de Musk", fontsize=12, fontweight="bold")
axes[1].set_xscale("log")
axes[1].legend(fontsize=10)
for bar, val_m in zip(hbars2, prof_s2["renda_vitalicia_brl"]/1e6):
    axes[1].text(val_m * 1.05, bar.get_y() + bar.get_height()/2,
                 f"R$ {val_m:.1f}M", va="center", fontsize=9)

axes[1].text(0.98, -0.08, "Fonte: Pisos/convenções coletivas 2025 | Forbes (jun/2026)",
             transform=axes[1].transAxes, fontsize=8, ha="right", color="gray")

plt.suptitle("Desigualdade: Renda Vitalícia de Trabalhadores Essenciais vs. Fortuna de Musk",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(DATA + "output_12_profissoes.png", dpi=150, bbox_inches="tight")
plt.show()


Renda vitalícia vs. Fortuna de Musk (35 anos de trabalho):
                     profissao  salario_medio_brl  renda_vitalicia_brl  n_vidas_para_musk
             Trabalhador Rural               1621               680820       8.386945e+06
           Auxiliar de Limpeza               1800               756000       7.552910e+06
   Agente Comunitário de Saúde               2700              1134000       5.035273e+06
Professor (Ensino Fundamental)               3200              1344000       4.248512e+06
         Técnico de Enfermagem               2800              1176000       4.855442e+06
                 Enfermeiro(a)               4750              1995000       2.862155e+06
     Bombeiro/Policial Militar               5500              2310000       2.471861e+06
       Professor Universitário               8500              3570000       1.599440e+06
                    Médico SUS              12000              5040000       1.132937e+06
                    Engenheiro           

In [2]:
print("=" * 65)
print("ANÁLISE: RENDA VITALÍCIA DE PROFISSÕES ESSENCIAIS")
print("=" * 65)
print(f"Vida laborativa considerada: {anos_trabalho} anos")
print(f"\n{'Profissão':35s} {'Salário/mês':>12s} {'Vida inteira':>15s} {'Vidas p/ Musk':>15s}")
print("-" * 80)
for _, row in prof.sort_values("n_vidas_para_musk", ascending=False).iterrows():
    print(f"{row['profissao']:35s} R$ {row['salario_medio_brl']:>8,.0f} "
          f"R$ {row['renda_vitalicia_brl']/1e6:>10.2f}M  {row['n_vidas_para_musk']:>15,.0f}")
print("\nFonte: Pisos/convenções coletivas 2025 | Forbes (jun/2026)")


ANÁLISE: RENDA VITALÍCIA DE PROFISSÕES ESSENCIAIS
Vida laborativa considerada: 35 anos

Profissão                            Salário/mês    Vida inteira   Vidas p/ Musk
--------------------------------------------------------------------------------
Trabalhador Rural                   R$    1,621 R$       0.68M        8,386,945
Auxiliar de Limpeza                 R$    1,800 R$       0.76M        7,552,910
Agente Comunitário de Saúde         R$    2,700 R$       1.13M        5,035,273
Técnico de Enfermagem               R$    2,800 R$       1.18M        4,855,442
Professor (Ensino Fundamental)      R$    3,200 R$       1.34M        4,248,512
Enfermeiro(a)                       R$    4,750 R$       2.00M        2,862,155
Bombeiro/Policial Militar           R$    5,500 R$       2.31M        2,471,861
Professor Universitário             R$    8,500 R$       3.57M        1,599,440
Engenheiro                          R$    9,500 R$       3.99M        1,431,078
Médico SUS                    